# Metrics Deep-Dive

The [cross-validation chapter](cross-validation.ipynb) reported **accuracy**, then
warned that accuracy hides a lot — especially under
[class imbalance](../01b-eda/exploratory-data-analysis.ipynb). This chapter opens
up the full metric catalogue and *visualises* it. The confusion-matrix plot reuses
the heatmap pattern from the
[visualization gallery](../01b-eda/visualization-gallery.ipynb); the ROC,
precision-recall and residual plots come from
[`plotters-statistical`](https://crates.io/crates/plotters-statistical), the same
crate the gallery uses for its statistical charts.

We reuse `smartcore`'s **breast cancer** dataset and a logistic-regression model,
fit once below; every metric in this notebook is computed on the same held-out
split so the numbers are directly comparable.

In [ ]:
:dep smartcore = { version = "0.3", features = ["datasets"] }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep plotters-statistical = { version = "0.2.0" }
use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linalg::basic::arrays::{Array, Array2};
use smartcore::dataset::breast_cancer;
use smartcore::model_selection::train_test_split;
use smartcore::linear::logistic_regression::LogisticRegression;
use plotters::prelude::*;

let (x, y): (DenseMatrix<f32>, Vec<i32>) = {
    let ds = breast_cancer::load_dataset();
    let x = DenseMatrix::new(ds.num_samples, ds.num_features, ds.data.clone(), false);
    let y = ds.target.iter().map(|&v| v as i32).collect();
    (x, y)
};

// Fit ONCE. Persist the held-out truth, predictions, and probability scores as
// plain Vecs (nameable types evcxr keeps across cells). `smartcore`'s logistic
// model has no `predict_proba`, so we recover the probability from the fitted
// coefficients: sigmoid(x·w + b). Coefficients are shape (1, n_features).
let (yte, pred, scores): (Vec<i32>, Vec<i32>, Vec<f32>) = {
    let (xtr, xte, ytr, yte) = train_test_split(&x, &y, 0.3, true, Some(42));
    let model = LogisticRegression::fit(&xtr, &ytr, Default::default()).unwrap();
    let pred = model.predict(&xte).unwrap();
    let coef = model.coefficients();
    let b = *model.intercept().get((0, 0));
    let (nte, nf) = xte.shape();
    let scores = (0..nte).map(|i| {
        let mut s = b;
        for j in 0..nf { s += *xte.get((i, j)) * *coef.get((0, j)); }
        1.0 / (1.0 + (-s).exp())
    }).collect();
    (yte, pred, scores)
};
println!("held-out: {} samples, {} positive", yte.len(), yte.iter().filter(|&&v| v == 1).count());

## Classification metrics & the confusion matrix

Accuracy is `(TP + TN) / total`. The other three headline metrics come from the
**confusion matrix** — the 2×2 count of predicted-vs-actual:

- **Precision** `TP / (TP + FP)` — of what we *called* positive, how much was?
- **Recall** `TP / (TP + FN)` — of the *actual* positives, how many did we catch?
- **F1** — their harmonic mean, a single balance of the two.

In [ ]:
use smartcore::metrics::{accuracy, precision, recall, f1};

let yte_f: Vec<f32> = yte.iter().map(|&v| v as f32).collect();
let pred_f: Vec<f32> = pred.iter().map(|&v| v as f32).collect();
println!("accuracy  = {:.3}", accuracy(&yte, &pred));
println!("precision = {:.3}", precision(&yte_f, &pred_f));
println!("recall    = {:.3}", recall(&yte_f, &pred_f));
println!("f1        = {:.3}", f1(&yte_f, &pred_f, 1.0));

// Confusion matrix by hand: cm[actual][predicted].
let mut cm = [[0u32; 2]; 2];
for (a, p) in yte.iter().zip(pred.iter()) { cm[*a as usize][*p as usize] += 1; }
println!("\nconfusion matrix  [actual][pred]:");
println!("  actual 0:  TN={:3}  FP={:3}", cm[0][0], cm[0][1]);
println!("  actual 1:  FN={:3}  TP={:3}", cm[1][0], cm[1][1]);

### Confusion matrix as a heatmap

The same heatmap technique from the
[visualization gallery](../01b-eda/visualization-gallery.ipynb#3-correlation-heatmap),
now over the 2×2 confusion counts — darker = more samples in that cell:

In [ ]:
let maxc = cm.iter().flatten().cloned().max().unwrap_or(1) as f64;
evcxr_figure((440, 400), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("confusion matrix", ("sans-serif", 18))
        .margin(10).x_label_area_size(40).y_label_area_size(50)
        .build_cartesian_2d(0f64..2f64, 0f64..2f64)?;
    chart.configure_mesh().disable_mesh()
        .x_labels(2).y_labels(2)
        .x_desc("predicted").y_desc("actual")
        .x_label_formatter(&|v| if *v < 0.5 { "0".into() } else if *v < 1.5 { "1".into() } else { "".into() })
        .y_label_formatter(&|v| if *v < 0.5 { "0".into() } else if *v < 1.5 { "1".into() } else { "".into() })
        .draw()?;
    let labels = [["TN", "FP"], ["FN", "TP"]];
    for a in 0..2 {
        for p in 0..2 {
            let t = cm[a][p] as f64 / maxc;
            let color = RGBColor((255.0 * (1.0 - t)) as u8, (255.0 * (1.0 - t)) as u8, 255);
            chart.draw_series(std::iter::once(
                Rectangle::new([(p as f64, a as f64), (p as f64 + 1.0, a as f64 + 1.0)], color.filled())))?;
            let txt = if t > 0.5 { WHITE } else { BLACK };
            chart.draw_series(std::iter::once(
                Text::new(format!("{} {}", labels[a][p], cm[a][p]), (p as f64 + 0.5, a as f64 + 0.55),
                          ("sans-serif", 16).into_font().color(&txt))))?;
        }
    }
    Ok(())
})

## ROC curve & AUC

A classifier outputs a *score*; the decision threshold is a choice. The **ROC
curve** sweeps every threshold, plotting the true-positive rate against the
false-positive rate. **AUC** (area under it) summarises threshold-independent
ranking quality: 0.5 is random, 1.0 is perfect.

`smartcore` 0.3 ships `roc_auc_score` for the AUC *number*. For the curve itself,
`plotters-statistical`'s `RocCurve::from_scores` takes the scores and boolean
labels, builds the curve, shades the AUC area, and draws the random-chance
baseline — and its legend carries the AUC it computes, a handy cross-check against
`smartcore`'s.

In [ ]:
use smartcore::metrics::roc_auc_score;
use plotters_statistical::style::palette_color;
use plotters_statistical::RocCurve;

let auc = roc_auc_score(&yte_f, &scores);
println!("smartcore roc_auc_score = {:.3}", auc);

// RocCurve::from_scores wants scores as f64 and boolean labels (true = positive).
// (Shared with the PR cell below.)
let sc: Vec<f64> = scores.iter().map(|&s| s as f64).collect();
let lab: Vec<bool> = yte.iter().map(|&v| v == 1).collect();

evcxr_figure((460, 440), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("ROC curve", ("sans-serif", 16))
        .margin(10).x_label_area_size(36).y_label_area_size(40)
        .build_cartesian_2d(0f64..1f64, 0f64..1f64)?;
    chart.configure_mesh().x_desc("false positive rate").y_desc("true positive rate").draw()?;
    let curve = RocCurve::from_scores(&sc, &lab)?
        .color(palette_color(0)).stroke_width(2).with_baseline().shade_area(true);
    let label = curve.legend_label("logistic");
    chart.draw_series(std::iter::once(curve))?
        .label(label)
        .legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], palette_color(0)));
    chart.configure_series_labels().position(SeriesLabelPosition::LowerRight)
        .border_style(BLACK).background_style(WHITE.mix(0.85)).draw()?;
    Ok(())
})

## Precision-recall curve

When positives are rare (imbalanced data — recall the
[class-balance chart](../01b-eda/exploratory-data-analysis.ipynb)), the
precision-recall curve is often more informative than ROC, because it ignores the
easy true-negatives and focuses on the positive class. `PrecisionRecallCurve::from_scores`
builds it from the same scores/labels, draws the prevalence baseline, and reports
the average precision in its legend:

In [ ]:
use plotters_statistical::PrecisionRecallCurve;

evcxr_figure((460, 400), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("precision-recall curve", ("sans-serif", 16))
        .margin(10).x_label_area_size(36).y_label_area_size(40)
        .build_cartesian_2d(0f64..1f64, 0f64..1.05f64)?;
    chart.configure_mesh().x_desc("recall").y_desc("precision").draw()?;
    let curve = PrecisionRecallCurve::from_scores(&sc, &lab)?
        .color(palette_color(1)).stroke_width(2).with_baseline().shade_area(true);
    let label = curve.legend_label("logistic");
    chart.draw_series(std::iter::once(curve))?
        .label(label)
        .legend(|(x, y)| PathElement::new(vec![(x, y), (x + 18, y)], palette_color(1)));
    chart.configure_series_labels().position(SeriesLabelPosition::LowerLeft)
        .border_style(BLACK).background_style(WHITE.mix(0.85)).draw()?;
    Ok(())
})

## Regression metrics & residual plots

Regression needs different measures. On `smartcore`'s **diabetes** dataset with a
linear model:

- **MAE** — mean absolute error (same units as the target, robust to outliers).
- **MSE / RMSE** — squared error, penalises large misses; RMSE is back in target units.
- **R²** — fraction of variance explained (1.0 perfect, 0.0 = no better than the mean).
- **MAPE** — mean absolute *percentage* error (`smartcore` has no built-in, so by hand).

In [ ]:
use smartcore::dataset::diabetes;
use smartcore::linear::linear_regression::LinearRegression;
use smartcore::metrics::{mean_squared_error, mean_absolute_error, r2};

let (truth, est): (Vec<f32>, Vec<f32>) = {
    let ds = diabetes::load_dataset();
    let dx = DenseMatrix::new(ds.num_samples, ds.num_features, ds.data.clone(), false);
    let dy: Vec<f32> = ds.target.iter().map(|&v| v as f32).collect();
    let (xtr, xte, ytr, yte) = train_test_split(&dx, &dy, 0.3, true, Some(42));
    let model = LinearRegression::fit(&xtr, &ytr, Default::default()).unwrap();
    (yte, model.predict(&xte).unwrap())
};

let mse = mean_squared_error(&truth, &est);
let mae = mean_absolute_error(&truth, &est);
let mape = {
    let (mut s, mut c) = (0.0f64, 0.0f64);
    for (t, e) in truth.iter().zip(est.iter()) {
        if *t != 0.0 { s += ((t - e) / t).abs() as f64; c += 1.0; }
    }
    100.0 * s / c
};
println!("MAE  = {:.2}", mae);
println!("MSE  = {:.2}", mse);
println!("RMSE = {:.2}", mse.sqrt());
println!("R2   = {:.3}", r2(&truth, &est));
println!("MAPE = {:.1}%", mape);

### Residual plots

Two standard regression diagnostics: **predicted vs actual** (points should hug
the diagonal — a plain scatter) and **residuals vs predicted** via
`plotters-statistical`'s `ResidualPlot`, which adds a zero reference line and a
moving-average **trend** line. A curve or funnel in that trend signals a
mis-specified model or non-constant variance:

In [ ]:
use plotters_statistical::ResidualPlot;

let lo = truth.iter().chain(est.iter()).cloned().fold(f32::INFINITY, f32::min) as f64;
let hi = truth.iter().chain(est.iter()).cloned().fold(f32::NEG_INFINITY, f32::max) as f64;
let fitted: Vec<f64> = est.iter().map(|&e| e as f64).collect();
let resid: Vec<f64> = truth.iter().zip(est.iter()).map(|(t, e)| (t - e) as f64).collect();

evcxr_figure((820, 380), |root| {
    root.fill(&WHITE)?;
    let panels = root.split_evenly((1, 2));
    // predicted vs actual — plain plotters scatter
    {
        let mut ch = ChartBuilder::on(&panels[0])
            .caption("predicted vs actual", ("sans-serif", 15))
            .margin(8).x_label_area_size(34).y_label_area_size(44)
            .build_cartesian_2d(lo..hi, lo..hi)?;
        ch.configure_mesh().x_desc("actual").y_desc("predicted").draw()?;
        ch.draw_series(LineSeries::new(vec![(lo, lo), (hi, hi)], BLACK.mix(0.4)))?;
        ch.draw_series(truth.iter().zip(est.iter())
            .map(|(t, e)| Circle::new((*t as f64, *e as f64), 3, BLUE.mix(0.5).filled())))?;
    }
    // residuals vs predicted — plotters-statistical ResidualPlot (zero line + trend)
    {
        let rlim = resid.iter().cloned().fold(0.0, |m, v| f64::max(m, v.abs())) * 1.1;
        let mut ch = ChartBuilder::on(&panels[1])
            .caption("residuals vs predicted", ("sans-serif", 15))
            .margin(8).x_label_area_size(34).y_label_area_size(44)
            .build_cartesian_2d(lo..hi, (-rlim)..rlim)?;
        ch.configure_mesh().x_desc("predicted").y_desc("residual").draw()?;
        ch.draw_series(std::iter::once(ResidualPlot::from_residuals(&fitted, &resid)?.trend(true)))?;
    }
    Ok(())
})

## Which metric / chart for which question

Extending the gallery's [chart-selection table](../01b-eda/visualization-gallery.ipynb#which-chart-for-which-question)
with evaluation-specific entries:

| Your question | Reach for |
| --- | --- |
| Where is the classifier making its errors (which direction)? | **Confusion matrix** heatmap |
| How good is ranking, independent of threshold? | **ROC curve / AUC** |
| How does precision trade against recall (esp. imbalanced)? | **Precision-recall curve** |
| Are regression errors patterned (bias, heteroscedasticity)? | **Residual plots** |
| One-number regression quality | **RMSE** (units) or **R²** (variance explained) |

Next: [learning curves](learning-curves.ipynb) — using these metrics to diagnose
whether a model is under- or over-fitting, and whether more data would help.